In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages, BaseMessage

class SearchState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    user_query: str      # 经过LLM理解后的用户需求总结
    search_query: str    # 优化后用于Tavily API的搜索查询
    search_results: str  # Tavily搜索返回的结果
    final_answer: str    # 最终生成的答案
    step: str            # 标记当前步骤


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import SecretStr
from tavily import TavilyClient

# 加载 .env 文件中的环境变量
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
model_name = os.getenv("MODEL_NAME", "gpt-4o-mini")

if not api_key:
    raise ValueError("OPENAI_API_KEY 环境变量未设置")

# 初始化模型
# 我们将使用这个 llm 实例来驱动所有节点的智能
llm = ChatOpenAI(
    model=model_name,
    api_key=SecretStr(api_key),
    base_url=base_url,
    temperature=0.7
)
# 初始化Tavily客户端
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


In [ ]:
from typing import Any

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

UNDERSTAND_PROMPT = """
分析用户的查询："{user_message}"
请完成两个任务：
1. 简洁总结用户想要了解什么
2. 生成最适合搜索引擎的关键词（中英文均可，要精准）

格式：
理解：[用户需求总结]
搜索词：[最佳搜索关键词]"""


def understand_query_node(state: SearchState)  -> dict[str, str | list[str | dict[Any, Any]] | list[AIMessage]]:
    """步骤1：理解用户查询并生成搜索关键词"""
    user_message = state["messages"][-1].content
    
    understand_prompt = UNDERSTAND_PROMPT.format(user_message=user_message)

    response = llm.invoke([SystemMessage(content=understand_prompt)])
    response_text: str = response.content if isinstance(response.content, str) else ""
    
    # 解析LLM的输出，提取理解和搜索关键词
    user_summary = user_message  # 默认输出
    search_query = user_message   # 默认使用原始查询
    if "理解：" in response_text:
        user_summary = response_text.split("理解：")[1].split("搜索词：")[0].strip()
    if "搜索词：" in response_text:
        search_query = response_text.split("搜索词：")[1].strip()
    
    return {
        "user_query": user_summary,
        "search_query": search_query,
        "step": "understood",
        "messages": [AIMessage(content=f"我将为您搜索：{search_query}")]
    }


In [ ]:
from typing import Any


def tavily_search_node(state: SearchState) -> dict[Any, Any] | dict[str, str | list[AIMessage]]:
    """步骤2：使用Tavily API进行真实搜索"""
    
    search_query = state["search_query"]
    
    try:
        print(f"🔍 正在搜索: {search_query}")
        
        # 调用Tavily搜索API
        response = tavily_client.search(
            query=search_query,
            search_depth="basic",
            include_answer=True,
            include_raw_content=False,
            max_results=5
        )
        
        # 处理搜索结果
        search_results = ""
        
        # 优先使用Tavily的综合答案
        if response.get("answer"):
            search_results = f"综合答案：\n{response['answer']}\n\n"
        
        # 添加具体的搜索结果
        if response.get("results"):
            search_results += "相关信息：\n"
            for i, result in enumerate(response["results"][:3], 1):
                title = result.get("title", "")
                content = result.get("content", "")
                url = result.get("url", "")
                search_results += f"{i}. {title}\n{content}\n来源：{url}\n\n"
        
        if not search_results:
            search_results = "抱歉，没有找到相关信息。"
        
        return {
            "search_results": search_results,
            "step": "searched",
            "messages": [AIMessage(content=f"✅ 搜索完成！找到了相关信息，正在为您整理答案...")]
        }
        
    except Exception as e:
        error_msg = f"搜索时发生错误: {str(e)}"
        print(f"❌ {error_msg}")
        
        return {
            "search_results": f"搜索失败：{error_msg}",
            "step": "search_failed",
            "messages": [AIMessage(content="❌ 搜索遇到问题，我将基于已有知识为您回答")]
        }

In [ ]:
from typing import Any


from langchain_core.messages.ai import AIMessage


FALLBACK_PROMPT = """
搜索API暂时不可用，请基于您的知识回答用户的问题：

用户问题：{user_query}

请提供一个有用的回答，并说明这是基于已有知识的回答。
"""

ANSWER_PROMPT = """
基于以下搜索结果为用户提供完整、准确的答案：

用户问题：{user_query}

搜索结果：
{search_results}

请要求：
1. 综合搜索结果，提供准确、有用的回答
2. 如果是技术问题，提供具体的解决方案或代码
3. 引用重要信息的来源
4. 回答要结构清晰、易于理解
5. 如果搜索结果不够完整，请说明并提供补充建议
"""

def generate_answer_node(state: SearchState) -> dict[str, str | list[str | dict[Any, Any]] | list[AIMessage]]:
    """步骤3：基于搜索结果生成最终答案"""
    
    # 检查是否有搜索结果
    if state["step"] == "search_failed":
        # 如果搜索失败，基于LLM知识回答
        fallback_prompt = FALLBACK_PROMPT.format(user_query=state["user_query"])
        
        response = llm.invoke([SystemMessage(content=fallback_prompt)])

        return {
            "final_answer": response.content,
            "step": "completed",
            "messages": [AIMessage(content=response.content)]
        }
    
    # 基于搜索结果生成答案
    answer_prompt = ANSWER_PROMPT.format(user_query=state["user_query"], search_results=state["search_results"])
    response = llm.invoke([SystemMessage(content=answer_prompt)])
    
    return {
        "final_answer": response.content,
        "step": "completed",
        "messages": [AIMessage(content=response.content)]
    }

In [ ]:
from langgraph.constants import END, START
from langgraph.graph.state import StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Image, display


def create_search_assistant():
    workflow = StateGraph(SearchState)
    
    # 添加三个节点
    workflow.add_node("understand", understand_query_node)
    workflow.add_node("search", tavily_search_node)
    workflow.add_node("answer", generate_answer_node)
    
    # 设置线性流程
    workflow.add_edge(START, "understand")
    workflow.add_edge("understand", "search")
    workflow.add_edge("search", "answer")
    workflow.add_edge("answer", END)
    
    # 编译图
    memory = InMemorySaver()
    app = workflow.compile(checkpointer=memory)

    display(Image(app.get_graph().draw_mermaid_png()))
    
    return app

In [ ]:
from datetime import date
import time
from langchain_core.runnables.config import RunnableConfig


app = create_search_assistant()
user_input = "为什么今天的美股市场大跌？"

initial_state: SearchState = {
    "messages": [HumanMessage(content=user_input)],
    "user_query": "",
    "search_query": "",
    "search_results": "",
    "final_answer": "",
    "step": "start"
}

config: RunnableConfig = {"configurable": {"thread_id": f"search-session-{date.fromtimestamp(time.time())}"}}

for chunk in app.stream(input=initial_state, config=config):
    print(chunk)
    # if "messages" in chunk:
        # chunk["messages"][-1].pretty_print()
    # for node_name, node_output in output.items():
    #     if "messages" in node_output and node_output["messages"]:
    #         latest_message = node_output["messages"][-1]
    #         if isinstance(latest_message, AIMessage):
    #             if node_name == "understand":
    #                 print(f"🧠 理解阶段: {latest_message.content}")
    #             elif node_name == "search":
    #                 print(f"🔍 搜索阶段: {latest_message.content}")
    #             elif node_name == "answer":
    #                 print(f"\n💡 最终回答:\n{latest_message.content}")